# DJEMBE_LLM — Analyse des frappes de djembé

Objectif : transformer les rythmes de djembé en **données visualisables**, à la manière du reel sur le chant des oiseaux.

Pipeline du notebook :
1. Charger les 7 rythmes (audio)
2. Explorer un rythme (onde, spectrogramme, écoute)
3. Détecter les **onsets** (chaque frappe)
4. Extraire des **features** de timbre par frappe
5. Standardiser
6. Réduire la dimension (PCA + UMAP) et visualiser
7. **Clustering** non-supervisé → bass / tone / slap
8. Écouter les clusters pour vérifier
9. Visualisation finale

> Astuce : exécute les cellules une par une (Maj+Entrée). Les paramètres réglables sont en MAJUSCULES.

## 0. Imports & configuration

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import librosa, librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import umap

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 4)

# --- paramètres réglables ---
SR  = 22050        # fréquence d'échantillonnage
WIN = 0.080        # fenêtre d'analyse après chaque frappe (secondes)
AUDIO_DIR = 'audio'
OUT_DIR   = 'output'
os.makedirs(OUT_DIR, exist_ok=True)
print('Config OK — librosa', librosa.__version__)

## 1. Charger les 7 rythmes

On charge chaque fichier en mono à 22 050 Hz et on garde tout dans le dictionnaire `AUDIO` (clé = nom du rythme).

In [ ]:
wavs = sorted(f for f in os.listdir(AUDIO_DIR) if f.endswith('.wav'))
AUDIO = {}
for w in wavs:
    name = os.path.splitext(w)[0]
    y, _ = librosa.load(os.path.join(AUDIO_DIR, w), sr=SR, mono=True)
    AUDIO[name] = y
    print(f'{name:14s}  {len(y)/SR:6.1f} s')
print('\nRythmes chargés :', list(AUDIO))

## 2. Explorer un rythme : onde + spectrogramme + écoute

On prend un **passage d'une minute** (et non le tout début, souvent occupé par des réglages/annonces). Règle `RYTHME`, `DEBUT` et `DUREE`.

In [ ]:
RYTHME = 'Djansa'   # <- essaie 'Woloso', 'Gede', ...
DEBUT  = 30         # début de l'extrait en secondes (saute l'intro/réglages)
DUREE  = 60         # longueur de l'extrait en secondes

y = AUDIO[RYTHME]
a = int(DEBUT*SR)
b = min(len(y), int((DEBUT+DUREE)*SR))
extrait = y[a:b]
print(f'{RYTHME} : extrait de {a/SR:.0f}s à {b/SR:.0f}s  ({(b-a)/SR:.0f}s sur {len(y)/SR:.0f}s au total)')

fig, ax = plt.subplots(2, 1, figsize=(14, 6))
librosa.display.waveshow(extrait, sr=SR, ax=ax[0])
ax[0].set(title=f'{RYTHME} — forme d\'onde ({(b-a)/SR:.0f}s, à partir de {DEBUT}s)')
S = librosa.feature.melspectrogram(y=extrait, sr=SR, n_mels=128)
librosa.display.specshow(librosa.power_to_db(S, ref=np.max), sr=SR,
                         x_axis='time', y_axis='mel', ax=ax[1])
ax[1].set(title='Mel-spectrogramme')
plt.tight_layout(); plt.show()
display(Audio(extrait, rate=SR))

## 3. Détecter les frappes (onsets)

L'**onset detection** repère les instants où une frappe commence. Les paramètres `delta` (sensibilité) et `wait` (écart minimum entre 2 frappes) changent le nombre de frappes détectées — à régler à l'œil et à l'oreille.

On visualise sur le même extrait d'une minute que la cellule 2.

In [ ]:
def detect_onsets(y, delta=0.2, wait=10):
    f = librosa.onset.onset_detect(y=y, sr=SR, backtrack=True, units='frames',
                                   pre_max=20, post_max=20, pre_avg=100,
                                   post_avg=100, delta=delta, wait=wait)
    return librosa.frames_to_time(f, sr=SR)

onsets = detect_onsets(extrait)
print(f'{len(onsets)} frappes détectées sur {(b-a)/SR:.0f}s '
      f'(~{len(onsets)/((b-a)/SR):.1f} frappes/s)')

# zoom sur 8s pour bien voir les marqueurs
z = extrait[:SR*8]; oz = onsets[onsets < 8]
plt.figure(figsize=(14, 3))
librosa.display.waveshow(z, sr=SR, alpha=0.6)
plt.vlines(oz, -1, 1, color='r', alpha=0.8, label='frappe')
plt.legend(); plt.title(f'{RYTHME} — onsets détectés (zoom 8s)'); plt.show()

## 4. Extraire les features de timbre par frappe

Pour chaque frappe on prend une petite fenêtre (`WIN`) et on calcule un vecteur décrivant le **timbre** :
- centroïde / bande passante / rolloff spectral (grave ↔ aigu)
- flatness (son tonal ↔ bruité → utile pour le slap)
- taux de passage par zéro, énergie (RMS)
- 13 MFCC (signature de timbre)

In [ ]:
def stroke_features(seg):
    mfcc = librosa.feature.mfcc(y=seg, sr=SR, n_mfcc=13).mean(axis=1)
    feats = {
        'centroid':  librosa.feature.spectral_centroid(y=seg, sr=SR).mean(),
        'bandwidth': librosa.feature.spectral_bandwidth(y=seg, sr=SR).mean(),
        'rolloff':   librosa.feature.spectral_rolloff(y=seg, sr=SR).mean(),
        'flatness':  librosa.feature.spectral_flatness(y=seg).mean(),
        'zcr':       librosa.feature.zero_crossing_rate(seg).mean(),
        'rms':       librosa.feature.rms(y=seg).mean(),
    }
    feats.update({f'mfcc{i}': v for i, v in enumerate(mfcc)})
    return feats

def extract_strokes(name, y, delta=0.2, wait=10):
    rows, wl = [], int(WIN*SR)
    for t in detect_onsets(y, delta, wait):
        s = int(t*SR); seg = y[s:s+wl]
        if len(seg) < wl//2:
            continue
        row = {'rhythm': name, 'time': float(t)}
        row.update(stroke_features(seg))
        rows.append(row)
    return rows

all_rows = []
for name, y in AUDIO.items():
    r = extract_strokes(name, y)
    print(f'{name:14s} : {len(r):4d} frappes')
    all_rows += r
df = pd.DataFrame(all_rows)
print('\nTotal :', len(df), 'frappes'); df.head()

## 5. Standardiser la matrice de features

Les features ont des échelles très différentes (Hz vs MFCC) : on centre/réduit pour que chacune compte autant.

In [ ]:
feat_cols = ['centroid','bandwidth','rolloff','flatness','zcr','rms'] + \
            [f'mfcc{i}' for i in range(13)]
X = StandardScaler().fit_transform(df[feat_cols].values)
print('Matrice X :', X.shape)

## 6. Réduction de dimension : PCA puis UMAP

On passe de 19 dimensions à 2 pour pouvoir dessiner. PCA = rapide/linéaire ; UMAP = non-linéaire, fait mieux ressortir les « paquets » (comme dans la visu des oiseaux).

In [ ]:
pca = PCA(n_components=2).fit_transform(X)
emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X)
df['umap_x'], df['umap_y'] = emb[:,0], emb[:,1]

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(pca[:,0], pca[:,1], s=5, alpha=0.5); ax[0].set_title('PCA')
ax[1].scatter(emb[:,0], emb[:,1], s=5, alpha=0.5); ax[1].set_title('UMAP')
plt.show()

## 7. Clustering bass / tone / slap

On commence par chercher le bon nombre de clusters avec le **score de silhouette**, puis on lance KMeans.

In [ ]:
for k in range(2, 7):
    lab = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    print(f'k={k} : silhouette = {silhouette_score(X, lab):.3f}')

In [ ]:
K = 3   # <- les 3 frappes de base du djembé
km = KMeans(n_clusters=K, n_init=10, random_state=0).fit(X)
df['cluster'] = km.labels_

# nommer les clusters selon le centroïde spectral moyen (grave->aigu)
order = df.groupby('cluster')['centroid'].mean().sort_values().index.tolist()
names = {order[0]:'bass', order[1]:'tone', order[2]:'slap'} if K==3 else \
        {c:f'c{c}' for c in order}
df['stroke'] = df['cluster'].map(names)
print(df['stroke'].value_counts())

## 8. Écouter les clusters (vérification à l'oreille)
Étape clé : on enchaîne quelques frappes d'un cluster pour entendre si « bass » sonne vraiment grave, etc.

In [ ]:
def ecouter(stroke, n=12, dur=0.25):
    sub = df[df['stroke']==stroke].sample(min(n, (df['stroke']==stroke).sum()),
                                          random_state=1)
    clips = []
    for _, r in sub.iterrows():
        y = AUDIO[r['rhythm']]; s = int(r['time']*SR)
        clips.append(y[s:s+int(dur*SR)])
    montage = np.concatenate(clips) if clips else np.zeros(SR)
    print(f'>>> {stroke} ({len(sub)} frappes enchaînées)')
    display(Audio(montage, rate=SR))

for s in ['bass','tone','slap']:
    ecouter(s)

## 9. Visualisation finale

À gauche : coloré par type de frappe. À droite : coloré par rythme.

In [ ]:
COLORS = {'bass':'#2b6cff','tone':'#d63bd6','slap':'#2ec45f'}
fig, ax = plt.subplots(1, 2, figsize=(15, 7), facecolor='#0d0d0d')
for a in ax: a.set_facecolor('#0d0d0d'); a.tick_params(colors='#555')

for s, c in COLORS.items():
    m = df['stroke']==s
    ax[0].scatter(df.loc[m,'umap_x'], df.loc[m,'umap_y'], s=6, c=c, alpha=.6, label=s)
ax[0].set_title('Par type de frappe', color='white')
ax[0].legend(markerscale=3, labelcolor='white', facecolor='#1a1a1a')

cmap = plt.get_cmap('turbo'); rs = sorted(df['rhythm'].unique())
for i, r in enumerate(rs):
    m = df['rhythm']==r
    ax[1].scatter(df.loc[m,'umap_x'], df.loc[m,'umap_y'], s=6,
                  c=[cmap(i/(len(rs)-1))], alpha=.6, label=r)
ax[1].set_title('Par rythme', color='white')
ax[1].legend(markerscale=3, labelcolor='white', facecolor='#1a1a1a')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/umap_notebook.png', dpi=140, facecolor='#0d0d0d')
plt.show()

## 10. Sauvegarde

On enregistre le tableau des frappes (features + coordonnées UMAP + cluster) pour la suite (visu interactive, classifieur de rythme...).

In [ ]:
df.to_csv(f'{OUT_DIR}/strokes.csv', index=False)
print('Sauvegardé ->', f'{OUT_DIR}/strokes.csv', '|', len(df), 'frappes')